In [ ]:
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "REDACTED_FOR_PUBLIC_PORTFOLIO"
password = "REDACTED_FOR_PUBLIC_PORTFOLIO"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)
    
## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'my-image.png' # replace with your own image
try:
    encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()
    logo_component = html.Img(src=f"data:image/png;base64,{encoded_image}",
                              style={"height":"60px","margin":"8px"})
except Exception:
    logo_component = html.Div() # if no logo file detected

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))
    
app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Div([
        logo_component,
        html.H2("Gordon Price Project 2 Dashboard", style={"margin":"6px 0 0 6px"})
    ], style={"display":"flex","alignItems":"center","gap":"12px"}),
    
    html.Hr(),
    
    dcc.RadioItems(
    id='filter-type',
    options=[
        {'label': 'Reset (All)', 'value': 'reset'},
        {'label': 'Water Rescue', 'value': 'water'},
        {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
        {'label': 'Disaster/Individual Tracking', 'value': 'disaster'},
    ],
        value='reset',
        inline=True,
        labelStyle={'marginRight': '18px', 'fontWeight': 500}
    ),
    
    html.Hr(),
    

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        row_selectable='single',
        selected_rows=[0],
        page_size=10,
        filter_action='native',
        sort_action='native',
        sort_mode='multi',
        column_selectable='single',
        style_table={'height': '400px', 'overflowY': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '120px', 'width': '120px', 'maxWidth': '180px',
            'whiteSpace': 'normal'
        }
    ),
        
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 

    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display': 'flex', 'gap': '16px'},    
             children=[
                 html.Div(id='graph-id', className='col s12 m6', style={'flex': 1}),
                 html.Div(id='map-id', className='col s12 m6', style={'flex': 1}),
             ]
            )
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    if filter_type == 'water':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Labrador Retriever", "Chesapeake Bay Retriever", "Newfoundland", "Labrador Retriever Mix"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'mountain':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
    else:
        query = {}
    
    dff = pd.DataFrame.from_records(db.read(query))
    if not dff.empty and '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    return dff.to_dict('records')

        
## FIX ME Add code to filter interactive data table with MongoDB queries
#
#        
#        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#        data=df.to_dict('records')
#       
#       
#        return (data,columns)
@app.callback(Output('datatable-id', 'style_data_conditional'),
              Input('datatable-id', 'selected_columns')
             )
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{'if': {'column_id': c}, 'background_color': '#D2F3FF'} for c in selected_columns]
# Display the breeds of animal based on quantity represented in
# the data table


@app.callback(
    Output('graph-id', "children"),
    Input('datatable-id', "derived_virtual_data"),
)
def update_graphs(viewData):
    if not viewData:
        return [html.Div("No data to chart")]
    dff = pd.DataFrame(viewData)
    if dff.empty or 'breed' not in dff.columns:
        return [html.Div("No data to chart")]
    top = dff['breed'].value_counts().nlargest(10)
    fig = px.pie(values=top.values, names=top.index,
                            title="Top Breeds in Current Selection (Top 10)")
    return [dcc.Graph(figure=fig)]
    
#This callback will highlight a cell on the data table when the user selects it
# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)

def update_map(viewData, index):
    if not viewData:
        return [html.Div("No data to chart")]
    
    dff = pd.DataFrame(viewData)
    if dff.empty:
        return [html.Div("No data to display")]
    # Because we only allow single row selection, the list can be converted to a row index here
    if not index:
        row = 0
    else: 
        row = index[0]
                   
    if row >= len(dff):
        return[html.Div("Invalid row index")]
        
    
    for col in ('location_lat', 'location_long'):
        if col not in dff.columns:
            return [html.Div(f"Column '{col}' not found in data")]
        if pd.isna(dff.iloc[row].get('location_lat')) or pd.isna(dff.iloc[row].get('location_long')):
            return [html.Div("Selected record is missing coordinates")]
        
    lat = float(dff.iloc[row]['location_lat'])
    lon = float(dff.iloc[row]['location_long'])
    breed = str(dff.iloc[row].get('breed', '(unknown)'))
    name = str(dff.iloc[row].get('name', '(no name)'))
    
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[30.75,-97.48], zoom=10, children=[
                   dl.TileLayer(id="base-layer-id"),
                   dl.Marker(position=[lat, lon], children=[
                       dl.Tooltip(breed),
                       dl.Popup([html.H4("Animal Name"), html.P(name)])
            ])
        ])
    ]



app.run_server()